In [2]:

from __future__ import annotations

from dataclasses import dataclass
from typing import Optional, Iterable, List
import csv
import re

import requests
from bs4 import BeautifulSoup


# Archivos de entrada/salida (ajústalos a tu gusto)
INPUT_TEXT_FILE = "summary-stomachneo-set.txt"
OUTPUT_CSV_FILE = "papers_info.csv"

EUROPE_PMC_BASE = "https://www.ebi.ac.uk/europepmc/webservices/rest"


@dataclass
class PaperInfo:
    doi: str
    title: Optional[str]
    abstract: Optional[str]
    methods: Optional[str]


# -------------------------
# 1. Extraer DOIs del texto
# -------------------------

def extract_dois_from_file(path: str) -> List[str]:
    with open(path, encoding="utf-8") as f:
        text = f.read()

    pattern = r"10\.\d{4,9}/[-._;()/:A-Za-z0-9]+"
    raw_dois = re.findall(pattern, text)

    # Limpiar por si vienen dentro de URLs tipo https://doi.org/10.xxxx
    cleaned = [doi.split("doi.org/")[-1] for doi in raw_dois]

    # Eliminar duplicados conservando el orden
    unique_dois = list(dict.fromkeys(cleaned))
    return unique_dois


# -------------------------------
# 2. Consultar metadata en Europe PMC
# -------------------------------

def fetch_europe_pmc_record(doi: str) -> Optional[dict]:
    params = {
        "query": f"DOI:{doi}",
        "resulttype": "core",
        "format": "json",
        "pageSize": 1,
    }
    try:
        resp = requests.get(f"{EUROPE_PMC_BASE}/search", params=params, timeout=20)
    except requests.RequestException:
        return None

    if not resp.ok:
        return None

    data = resp.json()
    results = data.get("resultList", {}).get("result", [])
    if not results:
        return None

    return results[0]


def get_fulltext_html_url(record: dict) -> Optional[str]:
    urls = record.get("fullTextUrlList", {}).get("fullTextUrl", [])
    for item in urls:
        if item.get("documentStyle") == "html":
            return item.get("url")
    return None


# ------------------------------
# 3. Descargar HTML y extraer métodos
# ------------------------------

def download_html(url: str) -> Optional[str]:
    try:
        resp = requests.get(url, timeout=30)
    except requests.RequestException:
        return None

    if not resp.ok:
        return None

    return resp.text


def extract_methods_from_html(html: str) -> Optional[str]:
    soup = BeautifulSoup(html, "html.parser")

    # Intento 1: usar headings y cortar sección "Methods"
    headings = soup.find_all(["h1", "h2", "h3", "h4"])
    for h in headings:
        heading_text = (h.get_text() or "").strip().lower()
        if "methods" in heading_text or "materials and methods" in heading_text:
            section_parts: List[str] = []
            for sibling in h.next_siblings:
                if getattr(sibling, "name", None) in ["h1", "h2", "h3", "h4"]:
                    break
                section_parts.append(sibling.get_text(separator="\n", strip=True))
            section_text = "\n".join(p for p in section_parts if p).strip()
            if section_text:
                return section_text

    # Intento 2: heurística sobre texto plano
    text = soup.get_text(separator="\n", strip=True)

    patterns = [
        r"(?is)(materials and methods)(.*?)(results|discussion|conclusion)",
        r"(?is)(methods)(.*?)(results|discussion|conclusion)",
    ]
    for pattern in patterns:
        match = re.search(pattern, text)
        if match:
            return match.group(0).strip()

    return None


# ------------------------------
# 4. Construir objeto PaperInfo
# ------------------------------

def build_paper_info(doi: str) -> PaperInfo:
    record = fetch_europe_pmc_record(doi)
    if record is None:
        return PaperInfo(doi=doi, title=None, abstract=None, methods=None)

    title = record.get("title")
    abstract = record.get("abstractText")

    html_url = get_fulltext_html_url(record)
    methods: Optional[str] = None

    if html_url:
        html = download_html(html_url)
        if html:
            methods = extract_methods_from_html(html)

    return PaperInfo(doi=doi, title=title, abstract=abstract, methods=methods)


# ------------------------------
# 5. Guardar resultados a CSV
# ------------------------------

def save_papers_to_csv(papers: Iterable[PaperInfo], path: str) -> None:
    fieldnames = ["doi", "title", "abstract", "methods"]
    with open(path, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for p in papers:
            writer.writerow(
                {
                    "doi": p.doi,
                    "title": p.title or "",
                    "abstract": p.abstract or "",
                    "methods": p.methods or "",
                }
            )


# ------------------------------
# 6. Main
# ------------------------------

def main() -> None:
    dois = extract_dois_from_file(INPUT_TEXT_FILE)

    papers: List[PaperInfo] = []
    for doi in dois:
        paper = build_paper_info(doi)
        papers.append(paper)

    save_papers_to_csv(papers, OUTPUT_CSV_FILE)


if __name__ == "__main__":
    main()
